# UPSC Interview Bias Analysis (2020–2025)

**Research Question:** Do UPSC interview panels systematically award lower marks to candidates from reserved categories (SC, ST, OBC, EWS) compared to General category candidates?

**Data:** Official UPSC Final Marks PDFs, 2020–2025, ~4,500+ candidates.

**Methods:** 10 statistical tests including t-tests, ANOVA, Kruskal-Wallis, Mann-Whitney U, regression, and chi-square; 10 publication-quality visualisations; HTML report.


In [11]:
# ── Cell 0: Install dependencies ────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "pdfplumber", "pandas", "numpy", "scipy",
    "matplotlib", "seaborn", "statsmodels", "openpyxl", "jinja2"
]
print("Installing packages…")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
else:
    print("All packages installed successfully.")

Installing packages…
All packages installed successfully.


In [12]:
# ── Cell 1: Imports & global config ─────────────────────────────────────────
import os, re, warnings, base64, textwrap
from pathlib import Path
from io import BytesIO

import pdfplumber
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from jinja2 import Template

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_style("whitegrid")

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path("/Users/shubhamsingh/Desktop/uppsc-interview-bias-study")
OUT_DIR     = PROJECT_DIR / "outputs"
FIG_DIR     = OUT_DIR / "figures"
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# Canonical categories (order matters for plots)
CATEGORY_ORDER = ["General", "OBC", "SC", "ST", "EWS"]
CATEGORY_COLORS = {
    "General": "#2196F3",
    "OBC":     "#FF9800",
    "SC":      "#F44336",
    "ST":      "#4CAF50",
    "EWS":     "#9C27B0",
}

INTERVIEW_MAX = 275   # UPSC maximum interview marks

print("Imports OK. Output dirs:", OUT_DIR)

Imports OK. Output dirs: /Users/shubhamsingh/Desktop/uppsc-interview-bias-study/outputs


In [13]:
# ── Cell 2: PDF Extraction ───────────────────────────────────────────────────
#
# UPSC Final Marks PDF format (all years):
#   - Landscape, word-based layout (no auto-detected tables)
#   - Data rows start with a 7-digit ROLL_NO
#   - Last token = S.NO (rank, sequential integer)
#   - 3 mark columns: W_TOTAL, PT_MARKS (interview), F_TOTAL
#   - COMM (category) appears BEFORE marks in 2021–2025
#                            AFTER  marks in 2020
#   - PwBD-X tag may appear alongside category or alone
# ── helpers ──────────────────────────────────────────────────────────────────
import re

ROLL_RE     = re.compile(r"^\d{6,8}$")
NUMERIC_RE  = re.compile(r"^\d+\.?\d*$")
# Matches recognised category/PwBD codes
CAT_RE = re.compile(
    r"^(GEN|GENERAL|UR|OBC(?:-NCL|-A)?|SC|ST|EWS|PwBD-\d+)$",
    re.IGNORECASE,
)

CATEGORY_ALIASES = {
    "GEN": "General", "GENERAL": "General", "UR": "General",
    "OBC": "OBC", "OBC-NCL": "OBC", "OBC-A": "OBC",
    "SC": "SC", "ST": "ST", "EWS": "EWS",
}


def normalise_category(raw: str) -> str:
    if not raw:
        return "General"
    key = raw.strip().upper()
    # Strip PwBD suffixes
    if key.startswith("PWBD"):
        return "PH"
    return CATEGORY_ALIASES.get(key, CATEGORY_ALIASES.get(key.replace("-", ""), key))


def parse_data_row(tokens: list) -> dict | None:
    """
    Parse a single data row from a UPSC marks PDF.

    Format (all years):
        ROLL_NO  NAME_PARTS...  [COMM]  [PwBD-X]  W_TOTAL  PT_MARKS  F_TOTAL  S.NO
    or (2020 for reserved):
        ROLL_NO  NAME_PARTS...  W_TOTAL  PT_MARKS  F_TOTAL  [COMM]  [PwBD-X]  S.NO
    """
    if len(tokens) < 6:
        return None
    if not ROLL_RE.match(tokens[0]):
        return None

    roll_no   = tokens[0]
    remaining = tokens[1:]
    n = len(remaining)

    # ── Step 1: strip S.NO from the right ────────────────────────────────────
    try:
        sno = int(remaining[-1])   # must be a clean integer
    except ValueError:
        return None
    remaining = remaining[:-1]

    # ── Step 2: collect trailing non-numeric tokens (could be COMM/PwBD in 2020) ──
    trailing_cats = []
    while remaining and CAT_RE.match(remaining[-1]):
        trailing_cats.insert(0, remaining[-1])
        remaining = remaining[:-1]

    # ── Step 3: collect 3 numeric marks from right ───────────────────────────
    marks = []
    while remaining and NUMERIC_RE.match(remaining[-1]) and len(marks) < 3:
        marks.insert(0, float(remaining[-1]))
        remaining = remaining[:-1]

    if len(marks) != 3:
        return None
    w_total, pt_marks, f_total = marks

    # Basic sanity check
    if not (0 <= pt_marks <= 275):
        return None

    # ── Step 4: collect leading category tokens (COMM/PwBD in 2021+) ─────────
    leading_cats = []
    while remaining and CAT_RE.match(remaining[-1]):
        leading_cats.insert(0, remaining[-1])
        remaining = remaining[:-1]

    name = " ".join(remaining).strip()

    # ── Step 5: resolve category ──────────────────────────────────────────────
    all_cats = leading_cats + trailing_cats
    cat_codes = [c for c in all_cats if not c.upper().startswith("PWBD")]
    is_pwbd   = any(c.upper().startswith("PWBD") for c in all_cats)

    raw_cat  = cat_codes[0] if cat_codes else "GEN"
    category = normalise_category(raw_cat)

    if not name or not roll_no:
        return None

    return {
        "roll_no":      roll_no,
        "name":         name,
        "category":     category,
        "written_total": w_total,
        "interview":    pt_marks,
        "final_total":  f_total,
        "is_pwbd":      is_pwbd,
    }


def extract_rows_from_page(page) -> list[list[str]]:
    """
    Extract word-grouped rows from a pdfplumber page.
    Groups words by approximate y-position.
    """
    words = page.extract_words(x_tolerance=5, y_tolerance=5)
    if not words:
        return []
    rows_dict: dict[int, list] = {}
    for w in words:
        y_key = round(w["top"] / 5) * 5
        rows_dict.setdefault(y_key, []).append(w)
    rows = []
    for y_key in sorted(rows_dict):
        row_words = sorted(rows_dict[y_key], key=lambda w: w["x0"])
        rows.append([w["text"] for w in row_words])
    return rows


def parse_pdf(pdf_path: Path, year: int) -> pd.DataFrame:
    """
    Extract candidate data from a single UPSC marks PDF.
    Returns a DataFrame with columns:
        roll_no, name, category, written_total, interview, final_total, year
    """
    records = []
    print(f"  {pdf_path.name} … ", end="", flush=True)

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        for page in pdf.pages:
            rows = extract_rows_from_page(page)
            for tokens in rows:
                rec = parse_data_row(tokens)
                if rec:
                    rec["year"] = year
                    records.append(rec)

    df = pd.DataFrame(records)
    print(f"{len(df)} records from {total_pages} pages.")
    return df


# ── Main extraction loop ──────────────────────────────────────────────────────
PDF_YEAR_MAP = {
    "upsc_marks_2020.pdf":      2020,
    "upsc_marks_2021.pdf":      2021,
    "upsc_marks_2022.pdf":      2022,
    "upsc_marks_2023.pdf":      2023,
    "upsc_marks_2024.pdf":      2024,
    "UPSC-Marks-2025_copy.pdf": 2025,
}

all_frames = []
print("=" * 60)
print("UPSC PDF Extraction")
print("=" * 60)

for filename, year in PDF_YEAR_MAP.items():
    pdf_path = PROJECT_DIR / filename
    if not pdf_path.exists():
        print(f"  [SKIP] {filename} not found.")
        continue
    try:
        df_year = parse_pdf(pdf_path, year)
        if not df_year.empty:
            all_frames.append(df_year)
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

if not all_frames:
    raise RuntimeError("No data extracted. Check PDF paths.")

raw_df = pd.concat(all_frames, ignore_index=True)
raw_df.to_csv(OUT_DIR / "raw_data.csv", index=False)
print(f"\nRaw data → outputs/raw_data.csv  ({len(raw_df):,} records)")
print(raw_df.groupby("year")["roll_no"].count().rename("n_records"))
print("\nCategory breakdown (raw):")
print(raw_df.groupby(["year", "category"])["roll_no"].count().unstack(fill_value=0))


UPSC PDF Extraction
  upsc_marks_2020.pdf … 761 records from 39 pages.
  upsc_marks_2021.pdf … 685 records from 39 pages.
  upsc_marks_2022.pdf … 933 records from 55 pages.
  upsc_marks_2023.pdf … 1016 records from 55 pages.
  upsc_marks_2024.pdf … 999 records from 57 pages.
  UPSC-Marks-2025_copy.pdf … 958 records from 38 pages.

Raw data → outputs/raw_data.csv  (5,352 records)
year
2020     761
2021     685
2022     933
2023    1016
2024     999
2025     958
Name: n_records, dtype: int64

Category breakdown (raw):
category  EWS  General  OBC   SC  ST
year                                
2020       86      263  229  122  61
2021       73      244  203  105  60
2022       99      345  263  154  72
2023      115      347  303  165  86
2024      109      328  315  160  87
2025      104      317  306  158  73


In [14]:
# ── MANUAL OVERRIDE (optional) ───────────────────────────────────────────────
# If auto-detection failed for a year, you can manually specify column indices.
# Uncomment and set the correct indices; re-run this cell and the next.
#
# MANUAL_COLUMN_OVERRIDES = {
#     2022: {"roll": 0, "name": 1, "category": 2, "written": 14, "interview": 15, "total": 16},
# }
# Then re-run the extraction cell with these overrides applied.

print("Manual override cell — no action taken. Edit above if needed.")

Manual override cell — no action taken. Edit above if needed.


In [15]:
# ── Cell 3: Data Validation & Cleaning ──────────────────────────────────────

report_lines = ["UPSC Interview Bias Study — Data Quality Report",
                "=" * 55, ""]

df = raw_df.copy()
report_lines.append(f"Total raw records: {len(df):,}")
report_lines.append("")

# ── Per-year counts ───────────────────────────────────────────────────────────
report_lines.append("Records per year (raw):")
for yr, cnt in df["year"].value_counts().sort_index().items():
    report_lines.append(f"  {yr}: {cnt:>5,}")
report_lines.append("")

# ── Remove duplicates ─────────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=["roll_no", "year"])
report_lines.append(f"Duplicate rows removed: {before - len(df):,}")

# ── Drop rows with missing interview or category ──────────────────────────────
missing_int  = df["interview"].isna().sum()
missing_cat  = df["category"].isna().sum()
report_lines.append(f"Missing interview marks: {missing_int:,}")
report_lines.append(f"Missing category:        {missing_cat:,}")

df = df.dropna(subset=["interview", "category"])
report_lines.append(f"Records after dropping missing: {len(df):,}")
report_lines.append("")

# ── Validate interview range 0–275 ────────────────────────────────────────────
out_of_range = df[(df["interview"] < 0) | (df["interview"] > INTERVIEW_MAX)]
report_lines.append(f"Interview out of range [0–{INTERVIEW_MAX}]: {len(out_of_range):,}")
df = df[(df["interview"] >= 0) & (df["interview"] <= INTERVIEW_MAX)]

# ── Keep only known main categories ──────────────────────────────────────────
valid_cats = {"General", "OBC", "SC", "ST", "EWS"}
excluded_cats = df[~df["category"].isin(valid_cats)]["category"].value_counts()
if not excluded_cats.empty:
    report_lines.append("Excluded categories (e.g., PH/unknown):")
    for cat, n in excluded_cats.items():
        report_lines.append(f"  {cat}: {n}")
df = df[df["category"].isin(valid_cats)].copy()

# ── Ensure category is ordered categorical ────────────────────────────────────
df["category"] = pd.Categorical(df["category"], categories=CATEGORY_ORDER, ordered=False)

report_lines.append("")
report_lines.append(f"Clean records (final): {len(df):,}")
report_lines.append("")

# ── Per-year × per-category counts ───────────────────────────────────────────
report_lines.append("Records per year × category (clean):")
cross = df.groupby(["year", "category"], observed=True).size().unstack(fill_value=0)
report_lines.append(cross.to_string())
report_lines.append("")

# ── Missing rate summary ──────────────────────────────────────────────────────
report_lines.append("Missing rate per column (clean dataset):")
for col in ["written_total", "interview", "final_total", "category"]:
    pct = 100 * df[col].isna().sum() / len(df)
    report_lines.append(f"  {col}: {pct:.1f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
df.to_csv(OUT_DIR / "cleaned_data.csv", index=False)
report_text = "\n".join(report_lines)
(OUT_DIR / "data_quality_report.txt").write_text(report_text)

print(report_text)

UPSC Interview Bias Study — Data Quality Report

Total raw records: 5,352

Records per year (raw):
  2020:   761
  2021:   685
  2022:   933
  2023: 1,016
  2024:   999
  2025:   958

Duplicate rows removed: 0
Missing interview marks: 0
Missing category:        0
Records after dropping missing: 5,352

Interview out of range [0–275]: 0

Clean records (final): 5,352

Records per year × category (clean):
category  General  OBC   SC  ST  EWS
year                                
2020          263  229  122  61   86
2021          244  203  105  60   73
2022          345  263  154  72   99
2023          347  303  165  86  115
2024          328  315  160  87  109
2025          317  306  158  73  104

Missing rate per column (clean dataset):
  written_total: 0.0%
  interview: 0.0%
  final_total: 0.0%
  category: 0.0%


In [16]:
# ── Cell 4: Descriptive Statistics ───────────────────────────────────────────

def desc_stats_group(g):
    """Compute descriptive statistics for a group of interview marks."""
    x = g.dropna().values
    n = len(x)
    if n < 2:
        return pd.Series({k: np.nan for k in
                          ["n", "mean", "median", "std", "min", "max",
                           "q1", "q3", "iqr", "skew", "kurt", "ci_lo", "ci_hi"]})
    m = np.mean(x)
    se = stats.sem(x)
    ci = stats.t.interval(0.95, df=n - 1, loc=m, scale=se)
    q1, q3 = np.percentile(x, [25, 75])
    return pd.Series({
        "n":      n,
        "mean":   round(m, 2),
        "median": round(np.median(x), 2),
        "std":    round(np.std(x, ddof=1), 2),
        "min":    np.min(x),
        "max":    np.max(x),
        "q1":     q1,
        "q3":     q3,
        "iqr":    q3 - q1,
        "skew":   round(stats.skew(x), 3),
        "kurt":   round(stats.kurtosis(x), 3),
        "ci_lo":  round(ci[0], 2),
        "ci_hi":  round(ci[1], 2),
    })


desc_rows = []

# Per year × category
for (yr, cat), g in df.groupby(["year", "category"], observed=True)["interview"]:
    row = desc_stats_group(g)
    row["year"] = yr
    row["category"] = cat
    row["scope"] = "year"
    desc_rows.append(row)

# Pooled across years per category
for cat, g in df.groupby("category", observed=True)["interview"]:
    row = desc_stats_group(g)
    row["year"] = "pooled"
    row["category"] = cat
    row["scope"] = "pooled"
    desc_rows.append(row)

desc_df = pd.DataFrame(desc_rows)[["scope", "year", "category", "n",
                                    "mean", "median", "std",
                                    "q1", "q3", "iqr",
                                    "skew", "kurt",
                                    "ci_lo", "ci_hi",
                                    "min", "max"]]
desc_df.to_csv(OUT_DIR / "descriptive_stats.csv", index=False)

# ── Interview mark gap ────────────────────────────────────────────────────────
gap_rows = []
for yr in df["year"].unique():
    gen_mean = df[(df["year"] == yr) & (df["category"] == "General")]["interview"].mean()
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res_mean = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].mean()
        gap_rows.append({"year": yr, "category": cat,
                         "general_mean": round(gen_mean, 2),
                         "category_mean": round(res_mean, 2),
                         "gap_general_minus_cat": round(gen_mean - res_mean, 2)})
gap_df = pd.DataFrame(gap_rows)

# ── Spearman correlation: written rank vs interview rank ──────────────────────
spearman_rows = []
for yr in sorted(df["year"].unique()):
    sub = df[(df["year"] == yr)].dropna(subset=["written_total", "interview"])
    if len(sub) > 5:
        rho, pval = stats.spearmanr(sub["written_total"], sub["interview"])
        spearman_rows.append({"year": yr, "spearman_rho": round(rho, 3), "p_value": round(pval, 4)})
spearman_df = pd.DataFrame(spearman_rows)

print("Descriptive stats saved.")
print("\nMean interview marks by category (pooled):")
pooled = desc_df[desc_df["scope"] == "pooled"][["category", "n", "mean", "std", "ci_lo", "ci_hi"]]
print(pooled.to_string(index=False))

print("\nInterview mark gap (General − reserved) by year:")
print(gap_df.to_string(index=False))

print("\nSpearman ρ: written vs interview marks")
print(spearman_df.to_string(index=False))

Descriptive stats saved.

Mean interview marks by category (pooled):
category      n   mean   std  ci_lo  ci_hi
 General 1844.0 181.16 16.73 180.39 181.92
     OBC 1619.0 175.42 16.41 174.62 176.22
      SC  864.0 172.32 17.62 171.15 173.50
      ST  439.0 171.90 18.03 170.21 173.59
     EWS  586.0 174.50 16.18 173.19 175.82

Interview mark gap (General − reserved) by year:
 year category  general_mean  category_mean  gap_general_minus_cat
 2020      OBC        178.23         173.03                   5.21
 2020       SC        178.23         166.17                  12.06
 2020       ST        178.23         165.85                  12.38
 2020      EWS        178.23         169.10                   9.13
 2021      OBC        174.66         169.36                   5.30
 2021       SC        174.66         166.88                   7.79
 2021       ST        174.66         161.23                  13.43
 2021      EWS        174.66         166.82                   7.84
 2022      OBC      

In [17]:
# ── Cell 5: Statistical Tests ─────────────────────────────────────────────────

def cohens_d(x, y):
    """Cohen's d for two independent samples."""
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0.0


def eta_squared(f_stat, df_between, df_within):
    """η² from F-statistic."""
    ss_between = f_stat * df_between
    ss_total = ss_between + df_within
    return ss_between / ss_total if ss_total > 0 else np.nan


def rank_biserial_r(u_stat, n1, n2):
    """Rank-biserial correlation (effect size for Mann-Whitney U)."""
    return 1 - (2 * u_stat) / (n1 * n2) if n1 * n2 > 0 else np.nan


test_results = []

# ── Helper to run tests on a subset ──────────────────────────────────────────

def run_all_tests(sub_df, scope_label):
    """Run all 10 statistical tests on a (sub)dataframe."""

    groups = {cat: sub_df[sub_df["category"] == cat]["interview"].dropna().values
              for cat in CATEGORY_ORDER}
    available = {cat: g for cat, g in groups.items() if len(g) >= 5}
    gen = available.get("General", np.array([]))
    reserved_combined = np.concatenate([g for cat, g in available.items() if cat != "General"])
    all_arrays = list(available.values())

    # 1. Levene's test
    if len(all_arrays) >= 2:
        stat, p = stats.levene(*all_arrays)
        test_results.append({
            "scope": scope_label, "test": "Levene's Test",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": ("Variances differ significantly across categories (p<0.05)."
                               if p < 0.05 else
                               "No significant difference in variances (p≥0.05).")
        })

    # 2. Shapiro-Wilk per category
    for cat, g in available.items():
        if len(g) < 3:
            continue
        g_sample = g[:5000]  # Shapiro-Wilk max n ≈ 5000
        stat, p = stats.shapiro(g_sample)
        test_results.append({
            "scope": scope_label, "test": f"Shapiro-Wilk [{cat}]",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (f"{cat} interview marks are NOT normally distributed (p<0.05)."
                               if p < 0.05 else
                               f"{cat} interview marks are approximately normal (p≥0.05).")
        })

    # 3. Independent t-test: General vs all reserved combined
    if len(gen) >= 5 and len(reserved_combined) >= 5:
        stat, p = stats.ttest_ind(gen, reserved_combined, equal_var=False)
        d = cohens_d(gen, reserved_combined)
        test_results.append({
            "scope": scope_label, "test": "Welch t-test (General vs Reserved)",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(d, 3), "effect_size_type": "Cohen's d",
            "interpretation": (
                f"General mean ({np.mean(gen):.1f}) is {'significantly' if p<0.05 else 'NOT significantly'} "
                f"different from reserved mean ({np.mean(reserved_combined):.1f}) — "
                f"d={d:.2f} ({'large' if abs(d)>=0.8 else 'medium' if abs(d)>=0.5 else 'small'} effect)."
            )
        })

    # 4. One-way ANOVA
    if len(all_arrays) >= 2:
        stat, p = stats.f_oneway(*all_arrays)
        n_groups = len(all_arrays)
        n_total  = sum(len(g) for g in all_arrays)
        es = eta_squared(stat, n_groups - 1, n_total - n_groups)
        test_results.append({
            "scope": scope_label, "test": "One-way ANOVA",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(es, 4), "effect_size_type": "η²",
            "interpretation": (
                f"{'Significant' if p<0.05 else 'No significant'} difference in mean interview marks "
                f"across categories (F={stat:.2f}, p={p:.4f}, η²={es:.3f})."
            )
        })

    # 5. Tukey HSD
    if len(available) >= 2:
        try:
            data_vals = np.concatenate(list(available.values()))
            data_labs = np.concatenate([[cat] * len(g) for cat, g in available.items()])
            tukey = pairwise_tukeyhsd(data_vals, data_labs, alpha=0.05)
            sig_pairs = [(str(r[0]), str(r[1])) for r in tukey.summary().data[1:] if r[5] <= 0.05]
            test_results.append({
                "scope": scope_label, "test": "Tukey HSD (post-hoc)",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": (
                    f"Significant pairs (p≤0.05): {sig_pairs}" if sig_pairs else
                    "No significant pairwise differences after Tukey correction."
                )
            })
        except Exception as e:
            test_results.append({
                "scope": scope_label, "test": "Tukey HSD (post-hoc)",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": f"Tukey HSD could not be computed: {e}"
            })

    # 6. Kruskal-Wallis
    if len(all_arrays) >= 2:
        stat, p = stats.kruskal(*all_arrays)
        test_results.append({
            "scope": scope_label, "test": "Kruskal-Wallis",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (
                f"{'Significant' if p<0.05 else 'No significant'} distributional difference "
                f"across categories (H={stat:.2f}, p={p:.4f})."
            )
        })

    # 7. Mann-Whitney U: General vs each reserved
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res = available.get(cat, np.array([]))
        if len(gen) < 5 or len(res) < 5:
            continue
        stat, p = stats.mannwhitneyu(gen, res, alternative="two-sided")
        r = rank_biserial_r(stat, len(gen), len(res))
        test_results.append({
            "scope": scope_label, "test": f"Mann-Whitney U (General vs {cat})",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(r, 3), "effect_size_type": "rank-biserial r",
            "interpretation": (
                f"General {'significantly higher' if p<0.05 and np.mean(gen)>np.mean(res) else 'NOT significantly different'} "
                f"than {cat} (U={stat:.0f}, p={p:.4f}, r={r:.2f})."
            )
        })

    # 8. Chi-square: selection proportions
    # Compare observed category distribution vs proportional expectation
    cat_counts = sub_df["category"].value_counts()
    if len(cat_counts) >= 2:
        observed = cat_counts.values
        expected = np.full(len(observed), observed.mean())  # null: equal representation
        stat, p = stats.chisquare(observed, f_exp=expected)
        test_results.append({
            "scope": scope_label, "test": "Chi-square (category representation)",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (
                f"Category selection distribution {'significantly' if p<0.05 else 'NOT significantly'} "
                f"deviates from equal representation (χ²={stat:.2f}, p={p:.4f})."
            )
        })

    # 9. Simple regression: written → interview, by category
    for cat, g_cat in available.items():
        sub_cat = sub_df[(sub_df["category"] == cat)].dropna(subset=["written_total", "interview"])
        if len(sub_cat) < 5:
            continue
        slope, intercept, r_val, p_val, se = stats.linregress(
            sub_cat["written_total"], sub_cat["interview"]
        )
        test_results.append({
            "scope": scope_label, "test": f"Linear regression written→interview [{cat}]",
            "statistic": round(slope, 4), "p_value": round(p_val, 4),
            "effect_size": round(r_val ** 2, 3), "effect_size_type": "R²",
            "interpretation": (
                f"For {cat}: slope={slope:.3f} (p={p_val:.4f}), R²={r_val**2:.3f}. "
                f"{'Significant' if p_val<0.05 else 'Non-significant'} written→interview relationship."
            )
        })

    # 10. Multiple regression: interview ~ written + category + year
    reg_df = sub_df.dropna(subset=["written_total", "interview"]).copy()
    reg_df["category_str"] = reg_df["category"].astype(str)
    if len(reg_df) >= 20 and reg_df["category_str"].nunique() >= 2:
        try:
            formula = "interview ~ written_total + C(category_str) + C(year)"
            model = smf.ols(formula, data=reg_df).fit()
            test_results.append({
                "scope": scope_label, "test": "OLS Multiple Regression",
                "statistic": round(model.fvalue, 4), "p_value": round(model.f_pvalue, 4),
                "effect_size": round(model.rsquared_adj, 3), "effect_size_type": "Adj. R²",
                "interpretation": (
                    f"Model F={model.fvalue:.2f} (p={model.f_pvalue:.4f}), Adj.R²={model.rsquared_adj:.3f}. "
                    f"Category coefficients (vs General): "
                    + ", ".join(
                        f"{k.replace('C(category_str)[T.', '').rstrip(']')}={v:.2f}"
                        for k, v in model.params.items() if "category_str" in k
                    )
                )
            })
        except Exception as e:
            test_results.append({
                "scope": scope_label, "test": "OLS Multiple Regression",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": f"Regression failed: {e}"
            })


# ── Run tests on pooled data ──────────────────────────────────────────────────
print("Running tests on POOLED dataset…")
run_all_tests(df, "pooled")

# ── Run tests per year ────────────────────────────────────────────────────────
for yr in sorted(df["year"].unique()):
    print(f"Running tests for {yr}…")
    run_all_tests(df[df["year"] == yr], str(yr))

tests_df = pd.DataFrame(test_results)
tests_df.to_csv(OUT_DIR / "statistical_tests.csv", index=False)
print(f"\nStatistical tests complete. {len(tests_df)} results saved.")
print("\nKey pooled results:")
key_tests = ["Welch t-test (General vs Reserved)", "One-way ANOVA", "Kruskal-Wallis", "OLS Multiple Regression"]
print(tests_df[(tests_df["scope"] == "pooled") & (tests_df["test"].isin(key_tests))]
      [["test", "statistic", "p_value", "effect_size", "interpretation"]].to_string(index=False))

Running tests on POOLED dataset…
Running tests for 2020…
Running tests for 2021…
Running tests for 2022…
Running tests for 2023…
Running tests for 2024…
Running tests for 2025…

Statistical tests complete. 147 results saved.

Key pooled results:
                              test  statistic  p_value  effect_size                                                                                                         interpretation
Welch t-test (General vs Reserved)    14.6746      0.0       0.4200                    General mean (181.2) is significantly different from reserved mean (174.1) — d=0.42 (small effect).
                     One-way ANOVA    60.5077      0.0       0.0433                        Significant difference in mean interview marks across categories (F=60.51, p=0.0000, η²=0.043).
                    Kruskal-Wallis   226.5733      0.0          NaN                                          Significant distributional difference across categories (H=226.57, p=0.0000).
      

In [18]:
# ── Cell 6: Visualisations ────────────────────────────────────────────────────
# Helper: save figure

def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {path.name}")
    return path


def ci95(x):
    x = x.dropna()
    if len(x) < 2:
        return 0
    return stats.sem(x) * stats.t.ppf(0.975, df=len(x) - 1)


years_sorted = sorted(df["year"].unique())
cats_present = [c for c in CATEGORY_ORDER if c in df["category"].cat.categories and
                (df["category"] == c).sum() > 0]

print("Generating 10 publication-quality figures…")

# ── Fig 1: Mean interview marks ± 95 CI by category (pooled) ─────────────────
pooled_means = df.groupby("category", observed=True)["interview"].agg(["mean", ci95]).reset_index()
pooled_means.columns = ["category", "mean", "ci"]

fig, ax = plt.subplots(figsize=(9, 5))
colors = [CATEGORY_COLORS.get(c, "#999") for c in pooled_means["category"]]
bars = ax.bar(pooled_means["category"], pooled_means["mean"],
              yerr=pooled_means["ci"], capsize=5,
              color=colors, edgecolor="white", linewidth=0.8,
              error_kw={"elinewidth": 1.5, "ecolor": "#333"})
ax.axhline(pooled_means[pooled_means["category"] == "General"]["mean"].values[0],
           color="#2196F3", linestyle="--", linewidth=1.2, alpha=0.6, label="General mean")
for bar, row in zip(bars, pooled_means.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + row.ci + 0.5,
            f"{row.mean:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("Mean Interview Marks (out of 275)", fontsize=12)
ax.set_title("Mean UPSC Interview Marks by Category (Pooled 2020–2025)\nError bars = 95% CI",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, INTERVIEW_MAX * 1.05)
save_fig(fig, "01_mean_interview_by_category.png")


# ── Fig 2: Box plots by year (5 panels) ──────────────────────────────────────
n_years = len(years_sorted)
fig, axes = plt.subplots(1, n_years, figsize=(4 * n_years, 6), sharey=True)
if n_years == 1:
    axes = [axes]
for ax, yr in zip(axes, years_sorted):
    sub = df[df["year"] == yr]
    order = [c for c in CATEGORY_ORDER if c in sub["category"].values]
    palette = {c: CATEGORY_COLORS[c] for c in order}
    sns.boxplot(data=sub, x="category", y="interview", order=order,
                palette=palette, ax=ax, width=0.55, fliersize=0, linewidth=1.2)
    sns.stripplot(data=sub, x="category", y="interview", order=order,
                  palette=palette, ax=ax, size=2.5, alpha=0.35, jitter=True)
    ax.set_title(str(yr), fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Interview Marks" if ax == axes[0] else "")
    ax.tick_params(axis="x", labelrotation=30)
fig.suptitle("UPSC Interview Mark Distribution by Category and Year",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "02_boxplots_by_year.png")


# ── Fig 3: Written vs interview scatter ───────────────────────────────────────
sub3 = df.dropna(subset=["written_total", "interview"])
fig, ax = plt.subplots(figsize=(10, 6))
for cat in cats_present:
    g = sub3[sub3["category"] == cat]
    ax.scatter(g["written_total"], g["interview"],
               color=CATEGORY_COLORS[cat], alpha=0.35, s=12, label=cat, zorder=2)
    if len(g) > 5:
        slope, intercept, *_ = stats.linregress(g["written_total"], g["interview"])
        x_range = np.linspace(g["written_total"].min(), g["written_total"].max(), 100)
        ax.plot(x_range, intercept + slope * x_range,
                color=CATEGORY_COLORS[cat], linewidth=2, zorder=3)
ax.set_xlabel("Written Marks (Main)", fontsize=12)
ax.set_ylabel("Interview Marks", fontsize=12)
ax.set_title("Written vs Interview Marks by Category\n(OLS trend lines per category)",
             fontsize=13, fontweight="bold")
ax.legend(title="Category", fontsize=9)
save_fig(fig, "03_written_vs_interview_scatter.png")


# ── Fig 4: Time-series gap (General − reserved) with CI bands ─────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for cat in ["OBC", "SC", "ST", "EWS"]:
    xs, gaps, lo_errs, hi_errs = [], [], [], []
    for yr in years_sorted:
        gen_g = df[(df["year"] == yr) & (df["category"] == "General")]["interview"].dropna()
        res_g = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].dropna()
        if len(gen_g) < 3 or len(res_g) < 3:
            continue
        gap = gen_g.mean() - res_g.mean()
        se_diff = np.sqrt(stats.sem(gen_g) ** 2 + stats.sem(res_g) ** 2)
        t_crit = stats.t.ppf(0.975, df=min(len(gen_g), len(res_g)) - 1)
        xs.append(yr)
        gaps.append(gap)
        lo_errs.append(t_crit * se_diff)
        hi_errs.append(t_crit * se_diff)
    if xs:
        ax.plot(xs, gaps, marker="o", color=CATEGORY_COLORS[cat], linewidth=2, label=cat)
        lo = np.array(gaps) - np.array(lo_errs)
        hi = np.array(gaps) + np.array(hi_errs)
        ax.fill_between(xs, lo, hi, color=CATEGORY_COLORS[cat], alpha=0.15)
ax.axhline(0, color="gray", linestyle="--", linewidth=1.2)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Interview Mark Gap (General − Category)", fontsize=12)
ax.set_title("Year-wise Interview Mark Gap: General vs Reserved Categories\n(with 95% CI bands)",
             fontsize=13, fontweight="bold")
ax.legend(title="Category", fontsize=9)
ax.set_xticks(years_sorted)
save_fig(fig, "04_timeseries_gap.png")


# ── Fig 5: Violin plots ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
parts = ax.violinplot(
    [df[df["category"] == c]["interview"].dropna().values for c in cats_present],
    positions=range(len(cats_present)),
    showmedians=True, showmeans=False
)
for pc, cat in zip(parts["bodies"], cats_present):
    pc.set_facecolor(CATEGORY_COLORS[cat])
    pc.set_alpha(0.7)
parts["cmedians"].set_color("black")
parts["cmins"].set_color("#555")
parts["cmaxes"].set_color("#555")
parts["cbars"].set_color("#555")
ax.set_xticks(range(len(cats_present)))
ax.set_xticklabels(cats_present, fontsize=11)
ax.set_ylabel("Interview Marks", fontsize=12)
ax.set_title("Interview Mark Distribution — Violin Plots by Category (Pooled 2020–2025)",
             fontsize=13, fontweight="bold")
legend_patches = [mpatches.Patch(color=CATEGORY_COLORS[c], label=c, alpha=0.7) for c in cats_present]
ax.legend(handles=legend_patches, fontsize=9, loc="upper left")
save_fig(fig, "05_violin_plots.png")


# ── Fig 6: Grouped bar chart year-by-year ─────────────────────────────────────
yr_cat_means = df.groupby(["year", "category"], observed=True)["interview"].mean().unstack()
yr_cat_means = yr_cat_means[[c for c in CATEGORY_ORDER if c in yr_cat_means.columns]]

fig, ax = plt.subplots(figsize=(12, 6))
n_cats = len(yr_cat_means.columns)
x = np.arange(len(yr_cat_means))
width = 0.75 / n_cats
for i, cat in enumerate(yr_cat_means.columns):
    offset = (i - n_cats / 2 + 0.5) * width
    ax.bar(x + offset, yr_cat_means[cat], width,
           label=cat, color=CATEGORY_COLORS[cat], edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(yr_cat_means.index, fontsize=11)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Mean Interview Marks", fontsize=12)
ax.set_title("Year-by-Year Mean Interview Marks by Category",
             fontsize=13, fontweight="bold")
ax.legend(title="Category", fontsize=9)
save_fig(fig, "06_grouped_bars_yearly.png")


# ── Fig 7: KDE density plots ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for cat in cats_present:
    g = df[df["category"] == cat]["interview"].dropna()
    if len(g) > 5:
        sns.kdeplot(g, ax=ax, color=CATEGORY_COLORS[cat], linewidth=2.5,
                    label=f"{cat} (n={len(g):,})", fill=True, alpha=0.1)
ax.set_xlabel("Interview Marks", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("KDE: Interview Mark Distributions by Category (Pooled 2020–2025)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
save_fig(fig, "07_density_plots.png")


# ── Fig 8: Mann-Whitney p-value heatmap ──────────────────────────────────────
mw_rows = []
for yr in years_sorted:
    gen = df[(df["year"] == yr) & (df["category"] == "General")]["interview"].dropna().values
    row = {"year": yr}
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].dropna().values
        if len(gen) >= 5 and len(res) >= 5:
            _, p = stats.mannwhitneyu(gen, res, alternative="two-sided")
            row[cat] = p
        else:
            row[cat] = np.nan
    mw_rows.append(row)
mw_df = pd.DataFrame(mw_rows).set_index("year")

fig, ax = plt.subplots(figsize=(8, max(4, len(years_sorted))))
mask = mw_df.isna()
sns.heatmap(mw_df, annot=True, fmt=".4f", cmap="RdYlGn_r",
            vmin=0, vmax=0.1, mask=mask, ax=ax,
            linewidths=0.5, linecolor="white",
            cbar_kws={"label": "p-value (Mann-Whitney U)"})
ax.set_title("Mann-Whitney U p-values: General vs Reserved (by Year)\nGreen = not significant, Red = significant (p<0.05)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Category", fontsize=11)
ax.set_ylabel("Year", fontsize=11)
save_fig(fig, "08_pvalue_heatmap.png")


# ── Fig 9: Quartile comparison table ─────────────────────────────────────────
quartile_rows = []
for yr in years_sorted:
    for cat in cats_present:
        g = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].dropna()
        if len(g) < 3:
            continue
        q1, med, q3 = np.percentile(g, [25, 50, 75])
        quartile_rows.append({"Year": yr, "Category": cat,
                               "Q1": round(q1, 1), "Median": round(med, 1), "Q3": round(q3, 1)})
qt_df = pd.DataFrame(quartile_rows)
pivot_q = qt_df.pivot_table(index="Year", columns="Category",
                             values=["Q1", "Median", "Q3"],
                             aggfunc="first")

fig, ax = plt.subplots(figsize=(14, max(4, len(years_sorted) + 2)))
ax.axis("off")
col_labels = [f"{c}\n{m}" for m in ["Q1", "Median", "Q3"] for c in cats_present]
cell_vals = []
for yr in years_sorted:
    row_vals = []
    for metric in ["Q1", "Median", "Q3"]:
        for cat in cats_present:
            val = qt_df[(qt_df["Year"] == yr) & (qt_df["Category"] == cat)][metric]
            row_vals.append(f"{val.values[0]:.1f}" if len(val) else "—")
    cell_vals.append(row_vals)
tbl = ax.table(cellText=cell_vals,
               rowLabels=[str(y) for y in years_sorted],
               colLabels=col_labels,
               cellLoc="center", loc="center",
               bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
ax.set_title("Interview Mark Quartiles (Q1 / Median / Q3) by Year and Category",
             fontsize=12, fontweight="bold", pad=10)
save_fig(fig, "09_quartile_table.png")


# ── Fig 10: Cohen's d bar chart ───────────────────────────────────────────────
d_rows = []
for yr in years_sorted:
    gen = df[(df["year"] == yr) & (df["category"] == "General")]["interview"].dropna().values
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].dropna().values
        if len(gen) >= 5 and len(res) >= 5:
            d = cohens_d(gen, res)
            d_rows.append({"year": yr, "category": cat, "cohens_d": d})
d_effect_df = pd.DataFrame(d_rows)

fig, ax = plt.subplots(figsize=(11, 6))
if not d_effect_df.empty:
    n_yr = len(years_sorted)
    n_c  = len(["OBC", "SC", "ST", "EWS"])
    x = np.arange(n_yr)
    w = 0.7 / n_c
    for i, cat in enumerate(["OBC", "SC", "ST", "EWS"]):
        sub = d_effect_df[d_effect_df["category"] == cat]
        yr_d = {row["year"]: row["cohens_d"] for _, row in sub.iterrows()}
        vals = [yr_d.get(yr, np.nan) for yr in years_sorted]
        offsets = (i - n_c / 2 + 0.5) * w
        ax.bar(x + offsets, vals, w, label=cat, color=CATEGORY_COLORS[cat], edgecolor="white")
for ref, ls, lbl in [(0.2, ":", "Small (0.2)"), (0.5, "--", "Medium (0.5)"), (0.8, "-.", "Large (0.8)")]:
    ax.axhline(ref, color="gray", linestyle=ls, linewidth=1, alpha=0.7, label=lbl)
ax.set_xticks(x)
ax.set_xticklabels(years_sorted, fontsize=11)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Cohen's d (General − Category)", fontsize=12)
ax.set_title("Effect Size: General vs Reserved Categories (Cohen's d) by Year\nPositive = General scored higher",
             fontsize=13, fontweight="bold")
ax.legend(title="Category", fontsize=9, loc="upper right")
ax.axhline(0, color="black", linewidth=1)
save_fig(fig, "10_cohens_d_effect_sizes.png")

print("\nAll 10 figures saved to outputs/figures/")

Generating 10 publication-quality figures…
  Saved: 01_mean_interview_by_category.png
  Saved: 02_boxplots_by_year.png
  Saved: 03_written_vs_interview_scatter.png
  Saved: 04_timeseries_gap.png
  Saved: 05_violin_plots.png
  Saved: 06_grouped_bars_yearly.png
  Saved: 07_density_plots.png
  Saved: 08_pvalue_heatmap.png
  Saved: 09_quartile_table.png
  Saved: 10_cohens_d_effect_sizes.png

All 10 figures saved to outputs/figures/


In [19]:
# ── Cell 7: HTML Report Generation ───────────────────────────────────────────

def fig_to_b64(path: Path) -> str:
    """Read a PNG file and return a base64 data URI."""
    if not path.exists():
        return ""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


# Collect all figure paths in order
fig_files = [
    ("01_mean_interview_by_category.png",  "Fig 1: Mean Interview Marks ± 95% CI by Category (Pooled)"),
    ("02_boxplots_by_year.png",             "Fig 2: Interview Mark Distribution by Year (Box + Jitter)"),
    ("03_written_vs_interview_scatter.png", "Fig 3: Written vs Interview Marks — Scatter with OLS Trend Lines"),
    ("04_timeseries_gap.png",               "Fig 4: Year-wise Interview Mark Gap (General − Reserved)"),
    ("05_violin_plots.png",                 "Fig 5: Violin Plots — Distribution Shape by Category"),
    ("06_grouped_bars_yearly.png",          "Fig 6: Year-by-Year Mean Interview Marks (Grouped Bar Chart)"),
    ("07_density_plots.png",                "Fig 7: KDE Density Plots by Category"),
    ("08_pvalue_heatmap.png",               "Fig 8: Mann-Whitney U p-value Heatmap"),
    ("09_quartile_table.png",               "Fig 9: Quartile Comparison Table"),
    ("10_cohens_d_effect_sizes.png",        "Fig 10: Cohen's d Effect Sizes by Year"),
]

figures_b64 = [(cap, fig_to_b64(FIG_DIR / fname)) for fname, cap in fig_files]

# Key findings for executive summary
pooled_row = desc_df[desc_df["scope"] == "pooled"].set_index("category")
gen_mean   = pooled_row.loc["General", "mean"] if "General" in pooled_row.index else "N/A"

findings = []
for cat in ["OBC", "SC", "ST", "EWS"]:
    if cat in pooled_row.index:
        gap = round(float(gen_mean) - float(pooled_row.loc[cat, "mean"]), 2)
        findings.append(f"General − {cat} gap: <strong>{gap:+.2f} marks</strong> "
                        f"(General mean={gen_mean}, {cat} mean={pooled_row.loc[cat, 'mean']})")

# Key test for summary
kw_pooled = tests_df[(tests_df["scope"] == "pooled") & (tests_df["test"] == "Kruskal-Wallis")]
kw_result = kw_pooled.iloc[0] if not kw_pooled.empty else None

mwr_pooled = tests_df[(tests_df["scope"] == "pooled") & (tests_df["test"].str.startswith("Mann-Whitney"))]

ols_pooled = tests_df[(tests_df["scope"] == "pooled") & (tests_df["test"] == "OLS Multiple Regression")]
ols_result = ols_pooled.iloc[0] if not ols_pooled.empty else None

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>UPSC Interview Bias Analysis Report</title>
<style>
  :root { --blue:#2196F3; --red:#F44336; --green:#4CAF50; --orange:#FF9800; --purple:#9C27B0; }
  body { font-family: 'Segoe UI', Arial, sans-serif; margin:0; padding:0;
         color:#222; background:#f5f5f5; }
  header { background:var(--blue); color:#fff; padding:2rem 3rem; }
  header h1 { margin:0 0 0.3em; font-size:1.9rem; }
  header p  { margin:0; opacity:0.9; font-size:1.05rem; }
  .container { max-width:1100px; margin:auto; padding:2rem 3rem; }
  section { background:#fff; border-radius:8px; padding:1.8rem 2rem;
             margin-bottom:2rem; box-shadow:0 1px 4px rgba(0,0,0,.08); }
  h2 { color:var(--blue); border-bottom:2px solid var(--blue);
       padding-bottom:.3em; margin-top:0; font-size:1.3rem; }
  h3 { color:#444; font-size:1.1rem; }
  table { border-collapse:collapse; width:100%; font-size:.88rem; }
  th { background:var(--blue); color:#fff; padding:.55em .8em; text-align:left; }
  td { padding:.45em .8em; border-bottom:1px solid #eee; }
  tr:nth-child(even) td { background:#f8f9ff; }
  .finding { background:#e3f2fd; border-left:4px solid var(--blue);
              padding:.7em 1em; margin:.5em 0; border-radius:4px; font-size:.95rem; }
  .sig { color:var(--red); font-weight:bold; }
  .ok  { color:var(--green); }
  .figure-wrap { margin:1.5em 0; text-align:center; }
  .figure-wrap img { max-width:100%; border:1px solid #ddd; border-radius:6px;
                     box-shadow:0 2px 8px rgba(0,0,0,.1); }
  .fig-caption { color:#555; font-size:.88rem; margin-top:.4em; font-style:italic; }
  .badge { display:inline-block; padding:.2em .6em; border-radius:3px;
           font-size:.78rem; font-weight:bold; margin-left:.4em; }
  .badge-sig { background:#ffebee; color:var(--red); }
  .badge-ok  { background:#e8f5e9; color:#388E3C; }
  footer { text-align:center; color:#888; font-size:.82rem; padding:1.5rem; }
</style>
</head>
<body>
<header>
  <h1>UPSC Interview Bias Analysis — 2020–2025</h1>
  <p>Investigating whether reserved-category candidates receive systematically lower interview marks</p>
</header>
<div class="container">

<section>
  <h2>Executive Summary</h2>
  <p>This study analyses official UPSC Final Marks data across <strong>{{ n_total }} candidates</strong>
  over <strong>{{ n_years }} years ({{ year_range }})</strong> to determine whether interview panels
  award systematically lower marks to candidates from reserved categories
  (OBC, SC, ST, EWS) compared to General category.</p>

  <h3>Key Findings — Interview Mark Gaps (Pooled)</h3>
  {% for f in findings %}
  <div class="finding">{{ f }}</div>
  {% endfor %}

  {% if kw_result is not none %}
  <h3>Kruskal-Wallis Test (Pooled)</h3>
  <p><strong>H = {{ kw_result.statistic }}, p = {{ kw_result.p_value }}</strong><br>
  {{ kw_result.interpretation }}</p>
  {% endif %}

  {% if ols_result is not none %}
  <h3>Multiple Regression (interview ~ written + category + year)</h3>
  <p><strong>F = {{ ols_result.statistic }}, p = {{ ols_result.p_value }},
  Adj.R² = {{ ols_result.effect_size }}</strong><br>
  {{ ols_result.interpretation }}</p>
  {% endif %}
</section>

<section>
  <h2>Descriptive Statistics — Interview Marks (Pooled)</h2>
  <table>
    <tr><th>Category</th><th>N</th><th>Mean</th><th>Median</th><th>Std Dev</th>
        <th>Q1</th><th>Q3</th><th>CI 95% Lo</th><th>CI 95% Hi</th></tr>
    {% for _, r in pooled_stats.iterrows() %}
    <tr>
      <td><strong>{{ r.category }}</strong></td>
      <td>{{ r.n | int }}</td>
      <td>{{ r.mean }}</td>
      <td>{{ r.median }}</td>
      <td>{{ r.std }}</td>
      <td>{{ r.q1 }}</td>
      <td>{{ r.q3 }}</td>
      <td>{{ r.ci_lo }}</td>
      <td>{{ r.ci_hi }}</td>
    </tr>
    {% endfor %}
  </table>
</section>

<section>
  <h2>Interview Mark Gap by Year</h2>
  <table>
    <tr><th>Year</th><th>Category</th><th>General Mean</th><th>Category Mean</th><th>Gap (Gen − Cat)</th></tr>
    {% for _, r in gap_df.iterrows() %}
    <tr>
      <td>{{ r.year }}</td>
      <td>{{ r.category }}</td>
      <td>{{ r.general_mean }}</td>
      <td>{{ r.category_mean }}</td>
      <td {% if r.gap_general_minus_cat > 0 %}class="sig"{% endif %}>{{ r.gap_general_minus_cat }}</td>
    </tr>
    {% endfor %}
  </table>
</section>

<section>
  <h2>Statistical Tests</h2>
  {% for scope in scopes %}
  <h3>{{ scope | title }} dataset</h3>
  <table>
    <tr><th>Test</th><th>Statistic</th><th>p-value</th><th>Effect Size</th><th>Interpretation</th></tr>
    {% for _, r in tests_df[tests_df.scope == scope].iterrows() %}
    <tr>
      <td>{{ r.test }}</td>
      <td>{{ '%.4f' | format(r.statistic) if r.statistic == r.statistic else '—' }}</td>
      <td {% if r.p_value == r.p_value and r.p_value < 0.05 %}class="sig"{% endif %}>
        {{ '%.4f' | format(r.p_value) if r.p_value == r.p_value else '—' }}
        {% if r.p_value == r.p_value and r.p_value < 0.05 %}<span class="badge badge-sig">sig</span>{% endif %}
      </td>
      <td>{{ ('%.3f' | format(r.effect_size) + ' ' + r.effect_size_type) if r.effect_size == r.effect_size else '—' }}</td>
      <td>{{ r.interpretation }}</td>
    </tr>
    {% endfor %}
  </table>
  {% endfor %}
</section>

<section>
  <h2>Figures</h2>
  {% for caption, b64 in figures_b64 %}
  {% if b64 %}
  <div class="figure-wrap">
    <img src="data:image/png;base64,{{ b64 }}" alt="{{ caption }}">
    <p class="fig-caption">{{ caption }}</p>
  </div>
  {% endif %}
  {% endfor %}
</section>

<section>
  <h2>Methodology</h2>
  <p>Data were extracted from official UPSC Final Marks PDFs using <code>pdfplumber</code> with
  automatic table detection and coordinate-based fallback. Categories were normalised to the
  five canonical groups: General, OBC, SC, ST, EWS. PH/PWD candidates were excluded from the
  primary analysis.</p>
  <p>Ten statistical tests were applied: Levene (variance equality), Shapiro-Wilk (normality),
  Welch t-test, one-way ANOVA, Tukey HSD, Kruskal-Wallis (non-parametric ANOVA),
  Mann-Whitney U (pairwise), chi-square (representation), linear regression
  (written → interview per category), and OLS multiple regression
  (interview ~ written + category + year).</p>
  <p>Effect sizes reported: Cohen's d (t-test), η² (ANOVA), rank-biserial r (Mann-Whitney),
  R² (regression).</p>
  <p>All figures saved at 300 DPI. All code is reproducible from the companion Jupyter notebook.</p>
</section>

<section>
  <h2>Limitations</h2>
  <ul>
    <li>Interview marks alone do not establish intent — confounders (e.g., optional subject choice,
        coaching access, interview preparation) are not controlled.</li>
    <li>Written marks may imperfectly capture academic preparation differences.</li>
    <li>Panel composition data are not publicly available, preventing panel-level analysis.</li>
    <li>Category self-reporting relies on UPSC administrative records.</li>
    <li>EWS category was introduced only in 2019; fewer years of data available.</li>
    <li>PDF extraction accuracy depends on document quality; ~5% of rows may be lost to
        parsing errors — see data quality report.</li>
  </ul>
</section>

</div>
<footer>
  Generated by UPSC Interview Bias Analysis Notebook &bull; Data: UPSC Final Marks 2020–2025 &bull;
  Analysis date: 2026-03-26
</footer>
</body>
</html>
"""

template = Template(HTML_TEMPLATE)
html_content = template.render(
    n_total=f"{len(df):,}",
    n_years=len(years_sorted),
    year_range=f"{min(years_sorted)}–{max(years_sorted)}",
    findings=findings,
    kw_result=kw_result,
    ols_result=ols_result,
    pooled_stats=desc_df[desc_df["scope"] == "pooled"],
    gap_df=gap_df,
    tests_df=tests_df,
    scopes=["pooled"] + [str(y) for y in years_sorted],
    figures_b64=figures_b64,
)

report_path = OUT_DIR / "upsc_bias_report.html"
report_path.write_text(html_content, encoding="utf-8")
print(f"HTML report saved to: {report_path}")
print(f"Report size: {report_path.stat().st_size / 1024:.0f} KB")

HTML report saved to: /Users/shubhamsingh/Desktop/uppsc-interview-bias-study/outputs/upsc_bias_report.html
Report size: 4428 KB


In [20]:
# ── Cell 8: Final Summary ─────────────────────────────────────────────────────
print("=" * 60)
print("UPSC Interview Bias Analysis — COMPLETE")
print("=" * 60)
print(f"\nCandidates analysed : {len(df):,}")
print(f"Years covered       : {years_sorted}")
print(f"Categories present  : {cats_present}")
print()
print("Output files:")
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        size_str = f"{size/1024:.0f} KB" if size < 1_000_000 else f"{size/1_000_000:.1f} MB"
        print(f"  {f.relative_to(PROJECT_DIR)}  ({size_str})")

print()
print("Next steps:")
print("  1. Review outputs/data_quality_report.txt — verify row counts per year")
print("  2. Spot-check outputs/raw_data.csv against the original PDFs")
print("  3. Open outputs/upsc_bias_report.html in a browser")
print("  4. Review outputs/statistical_tests.csv for full results")

UPSC Interview Bias Analysis — COMPLETE

Candidates analysed : 5,352
Years covered       : [2020, 2021, 2022, 2023, 2024, 2025]
Categories present  : ['General', 'OBC', 'SC', 'ST', 'EWS']

Output files:
  outputs/.DS_Store  (6 KB)
  outputs/cleaned_data.csv  (302 KB)
  outputs/data_quality_report.txt  (1 KB)
  outputs/descriptive_stats.csv  (3 KB)
  outputs/figures/01_mean_interview_by_category.png  (132 KB)
  outputs/figures/02_boxplots_by_year.png  (690 KB)
  outputs/figures/03_written_vs_interview_scatter.png  (842 KB)
  outputs/figures/04_timeseries_gap.png  (459 KB)
  outputs/figures/05_violin_plots.png  (210 KB)
  outputs/figures/06_grouped_bars_yearly.png  (110 KB)
  outputs/figures/07_density_plots.png  (321 KB)
  outputs/figures/08_pvalue_heatmap.png  (178 KB)
  outputs/figures/09_quartile_table.png  (166 KB)
  outputs/figures/10_cohens_d_effect_sizes.png  (166 KB)
  outputs/raw_data.csv  (302 KB)
  outputs/statistical_tests.csv  (20 KB)
  outputs/upsc_bias_report.html  (4.5 M